# Hospital Patient Visualizations & Dashboard

## Overview
This notebook creates interactive and static visualizations for the Hospital Patient Insights Dashboard:
- Interactive Plotly dashboards for stakeholder presentations
- Publication-quality Seaborn statistical visualizations
- Performance metrics and KPI visualizations
- Exportable PNG figures for reports

**Output**: High-quality visualizations saved to `../output/` directory

## 1. Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set styling
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Create output directory
output_dir = os.path.join('..', 'output')
os.makedirs(output_dir, exist_ok=True)

print("✓ Libraries loaded and output directory ready")

: 

## 2. Load Data

In [ ]:
DB_PATH = os.path.join('..', 'data', 'hospital.db')
conn = sqlite3.connect(DB_PATH)

# Load cleaned data if available, else raw
try:
    patients = pd.read_sql_query("SELECT * FROM patients_cleaned", conn)
    admissions = pd.read_sql_query("SELECT * FROM admissions_cleaned", conn)
    treatments = pd.read_sql_query("SELECT * FROM treatments_cleaned", conn)
except:
    patients = pd.read_sql_query("SELECT * FROM patients", conn)
    admissions = pd.read_sql_query("SELECT * FROM admissions", conn)
    treatments = pd.read_sql_query("SELECT * FROM treatments", conn)

# Prepare data
if 'admission_date' in admissions.columns:
    admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
    admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])
    if 'length_of_stay' not in admissions.columns:
        admissions['length_of_stay'] = (admissions['discharge_date'] - admissions['admission_date']).dt.days

print(f"✓ Data loaded: {len(patients):,} patients, {len(admissions):,} admissions, {len(treatments):,} treatments")

## 3. Patient Recovery Rates by Department

# Recovery rate calculation
dept_col = 'department_name' if 'department_name' in admissions.columns else 'department_id'

# Merge treatments with department info
treatment_dept = treatments.merge(
    admissions[['admission_id', dept_col]],
    left_on='admission_id',
    right_on='admission_id'
)

recovery_by_dept = treatment_dept.groupby(dept_col).apply(
    lambda x: (x['outcome'].isin(['Recovered', 'Improved'])).sum() / len(x) * 100
).sort_values(ascending=False)

# Plotly bar chart
fig = go.Figure(data=[
    go.Bar(x=recovery_by_dept.index, y=recovery_by_dept.values,
           marker=dict(color=recovery_by_dept.values,
                       colorscale='Viridis',
                       showscale=True))
])

fig.update_layout(
    title='Patient Recovery Rates by Department',
    xaxis_title='Department',
    yaxis_title='Recovery Rate (%)',
    height=500,
    showlegend=False,
    template='plotly_white'
)

fig.write_html(os.path.join(output_dir, '02_recovery_by_department.html'))
print("✓ Saved: 02_recovery_by_department.html")
fig.show()

# Seaborn visualization
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=recovery_by_dept.index, y=recovery_by_dept.values, palette='husl', ax=ax)
ax.set_title('Patient Recovery Rates by Department', fontsize=14, fontweight='bold')
ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Recovery Rate (%)', fontsize=12)
ax.axhline(y=recovery_by_dept.mean(), color='red', linestyle='--', label=f'Average: {recovery_by_dept.mean():.1f}%')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '02_recovery_by_department_sns.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 02_recovery_by_department_sns.png")

## 4. Average Length of Stay Trends

# Monthly trends
admissions['admission_month'] = admissions['admission_date'].dt.month
admissions['admission_year'] = admissions['admission_date'].dt.year
admissions['year_month'] = admissions['admission_date'].dt.to_period('M')

monthly_los = admissions.groupby('year_month')['length_of_stay'].mean()
monthly_los.index = monthly_los.index.to_timestamp()

# Plotly line chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly_los.index,
    y=monthly_los.values,
    mode='lines+markers',
    name='Average LOS',
    line=dict(color='#636EFA', width=3),
    marker=dict(size=8)
))

fig.add_hline(y=monthly_los.mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean: {monthly_los.mean():.2f} days",
              annotation_position="right")

fig.update_layout(
    title='Average Length of Stay - Monthly Trend',
    xaxis_title='Month',
    yaxis_title='Length of Stay (days)',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.write_html(os.path.join(output_dir, '03_los_trend.html'))
print("✓ Saved: 03_los_trend.html")
fig.show()

# Seaborn visualization with confidence interval
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_los.index, monthly_los.values, marker='o', linewidth=2.5, markersize=8, color='#636EFA')
ax.fill_between(monthly_los.index, monthly_los.values - monthly_los.std(), monthly_los.values + monthly_los.std(), 
                alpha=0.2, color='#636EFA')
ax.axhline(y=monthly_los.mean(), color='red', linestyle='--', label=f'Average: {monthly_los.mean():.2f} days')
ax.set_title('Average Length of Stay - Monthly Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Length of Stay (days)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '03_los_trend_sns.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 03_los_trend_sns.png")

# Department-wise LOS boxplot
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=admissions, x=dept_col, y='length_of_stay', palette='Set2', ax=ax)
ax.set_title('Length of Stay Distribution by Department', fontsize=14, fontweight='bold')
ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Length of Stay (days)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '04_los_by_department_box.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 04_los_by_department_box.png")

## 5. Department Performance Heatmap

# Create comprehensive department metrics
dept_metrics = treatment_dept.groupby(dept_col).agg({
    'admission_id': 'nunique',
    'treatment_type': 'count',
    'effectiveness_score': 'mean',
    'treatment_cost': 'mean'
}).rename(columns={
    'admission_id': 'Admissions',
    'treatment_type': 'Treatments',
    'effectiveness_score': 'Avg Effectiveness',
    'treatment_cost': 'Avg Cost'
})

# Add recovery rate
dept_metrics['Recovery Rate (%)'] = recovery_by_dept

# Normalize for heatmap
dept_metrics_normalized = dept_metrics.copy()
for col in dept_metrics_normalized.columns:
    dept_metrics_normalized[col] = (dept_metrics_normalized[col] - dept_metrics_normalized[col].min()) / \
                                    (dept_metrics_normalized[col].max() - dept_metrics_normalized[col].min())

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(dept_metrics_normalized.T, annot=dept_metrics.T, fmt='.1f', 
           cmap='RdYlGn', cbar_kws={'label': 'Normalized Score'}, ax=ax)
ax.set_title('Department Performance Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Department', fontsize=12)
ax.set_ylabel('Metrics', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '05_department_performance_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 05_department_performance_heatmap.png")

# Plotly heatmap
fig = go.Figure(data=go.Heatmap(
    z=dept_metrics_normalized.T.values,
    x=dept_metrics_normalized.index,
    y=dept_metrics_normalized.columns,
    colorscale='RdYlGn'
))

fig.update_layout(
    title='Department Performance Metrics Heatmap',
    xaxis_title='Department',
    yaxis_title='Metrics',
    height=400
)

fig.write_html(os.path.join(output_dir, '05_department_performance_heatmap.html'))
print("✓ Saved: 05_department_performance_heatmap.html")
fig.show()

# Admission volume trends
monthly_admissions = admissions.groupby('year_month').size()
monthly_admissions.index = monthly_admissions.index.to_timestamp()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly_admissions.index,
    y=monthly_admissions.values,
    fill='tozeroy',
    name='Admissions',
    line=dict(color='#00CC96', width=3)
))

fig.update_layout(
    title='Monthly Admission Volume',
    xaxis_title='Month',
    yaxis_title='Number of Admissions',
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.write_html(os.path.join(output_dir, '06_admission_volume_trend.html'))
print("✓ Saved: 06_admission_volume_trend.html")
fig.show()

# Seaborn admission trends
fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(monthly_admissions.index, monthly_admissions.values, alpha=0.3, color='#00CC96')
ax.plot(monthly_admissions.index, monthly_admissions.values, marker='o', linewidth=2.5, markersize=8, color='#00CC96')
ax.axhline(y=monthly_admissions.mean(), color='red', linestyle='--', 
           label=f'Average: {monthly_admissions.mean():.0f} admissions/month')
ax.set_title('Monthly Admission Volume Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Admissions', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '06_admission_volume_trend_sns.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 06_admission_volume_trend_sns.png")

# Treatment effectiveness comparison
treatment_effectiveness = treatments.groupby('treatment_type').agg({
    'effectiveness_score': 'mean',
    'treatment_id': 'count',
    'outcome': lambda x: (x.isin(['Recovered', 'Improved'])).sum() / len(x) * 100
}).rename(columns={
    'effectiveness_score': 'Effectiveness Score',
    'treatment_id': 'Count',
    'outcome': 'Recovery Rate (%)'
}).sort_values('Effectiveness Score', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=treatment_effectiveness.index,
    y=treatment_effectiveness['Effectiveness Score'],
    name='Effectiveness Score',
    marker_color='#AB63FA'
))

fig.add_trace(go.Scatter(
    x=treatment_effectiveness.index,
    y=treatment_effectiveness['Recovery Rate (%)'],
    name='Recovery Rate (%)',
    yaxis='y2',
    line=dict(color='#FF6692', width=3),
    mode='lines+markers'
))

fig.update_layout(
    title='Treatment Effectiveness Comparison',
    xaxis_title='Treatment Type',
    yaxis_title='Effectiveness Score',
    yaxis2=dict(title='Recovery Rate (%)', overlaying='y', side='right'),
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig.write_html(os.path.join(output_dir, '07_treatment_effectiveness.html'))
print("✓ Saved: 07_treatment_effectiveness.html")
fig.show()

# Seaborn treatment effectiveness
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(treatment_effectiveness))
bars = ax.bar(x_pos, treatment_effectiveness['Effectiveness Score'], color='#AB63FA', alpha=0.7)
ax2 = ax.twinx()
ax2.plot(x_pos, treatment_effectiveness['Recovery Rate (%)'], color='#FF6692', marker='o', linewidth=2.5, markersize=8, label='Recovery Rate')

ax.set_title('Treatment Type Effectiveness & Recovery Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Treatment Type', fontsize=12)
ax.set_ylabel('Effectiveness Score', fontsize=12, color='#AB63FA')
ax2.set_ylabel('Recovery Rate (%)', fontsize=12, color='#FF6692')
ax.set_xticks(x_pos)
ax.set_xticklabels(treatment_effectiveness.index, rotation=45, ha='right')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, '07_treatment_effectiveness_sns.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 07_treatment_effectiveness_sns.png")

# Resource utilization metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Treatments by department
dept_treatment_count = treatment_dept.groupby(dept_col).size().sort_values(ascending=True)
axes[0, 0].barh(dept_treatment_count.index, dept_treatment_count.values, color='#636EFA')
axes[0, 0].set_title('Treatments by Department', fontweight='bold')
axes[0, 0].set_xlabel('Count')

# 2. Average cost by treatment type
avg_cost_by_type = treatments.groupby('treatment_type')['treatment_cost'].mean().sort_values()
axes[0, 1].barh(avg_cost_by_type.index, avg_cost_by_type.values, color='#00CC96')
axes[0, 1].set_title('Average Cost by Treatment Type', fontweight='bold')
axes[0, 1].set_xlabel('Average Cost ($)')

# 3. Treatment outcome distribution
outcome_dist = treatments['outcome'].value_counts()
axes[1, 0].pie(outcome_dist.values, labels=outcome_dist.index, autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Treatment Outcomes Distribution', fontweight='bold')

# 4. LOS distribution by status
if 'status' in admissions.columns:
    admissions_status = admissions.dropna(subset=['status'])
    sns.boxplot(data=admissions_status, x='status', y='length_of_stay', ax=axes[1, 1], palette='Set2')
    axes[1, 1].set_title('LOS by Admission Status', fontweight='bold')
    axes[1, 1].set_xlabel('Status')
    axes[1, 1].set_ylabel('Length of Stay (days)')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, '08_resource_utilization.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: 08_resource_utilization.png")

# Summary dashboard with key metrics
fig = go.Figure()

# Calculate key metrics
total_patients = len(patients)
total_admissions = len(admissions)
avg_los = admissions['length_of_stay'].mean()
recovery_rate = (treatments['outcome'].isin(['Recovered', 'Improved'])).sum() / len(treatments) * 100
total_cost = treatments['treatment_cost'].sum()
avg_effectiveness = treatments['effectiveness_score'].mean()

# Create indicator metrics
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=total_patients,
    title={"text": "Total Patients"},
    domain={'x': [0, 0.25], 'y': [0.75, 1]}
))

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=total_admissions,
    title={"text": "Total Admissions"},
    domain={'x': [0.25, 0.5], 'y': [0.75, 1]}
))

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=avg_los,
    title={"text": "Avg Length of Stay (days)"},
    domain={'x': [0.5, 0.75], 'y': [0.75, 1]}
))

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=recovery_rate,
    title={"text": "Recovery Rate (%)"},
    gauge={'axis': {'range': [0, 100]},
            'bar': {'color': "darkblue"},
            'steps': [{'range': [0, 50], 'color': "lightgray"},
                      {'range': [50, 75], 'color': "gray"},
                      {'range': [75, 100], 'color': "lightgreen"}]},
    domain={'x': [0, 0.5], 'y': [0, 0.5]}
))

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=avg_effectiveness,
    title={"text": "Avg Effectiveness Score"},
    gauge={'axis': {'range': [0, 1]},
            'bar': {'color': "darkgreen"}},
    domain={'x': [0.5, 1], 'y': [0, 0.5]}
))

fig.update_layout(title="Hospital Dashboard - Key Metrics")
fig.write_html(os.path.join(output_dir, '09_kpi_dashboard.html'))
print("✓ Saved: 09_kpi_dashboard.html")
fig.show()

conn.close()

print("\n" + "=" * 70)
print("VISUALIZATION SUMMARY")
print("=" * 70)
print(f"\nAll visualizations saved to: {output_dir}")
print("\nGenerated Files:")
print("  HTML (Interactive):")
print("    - 02_recovery_by_department.html")
print("    - 03_los_trend.html")
print("    - 05_department_performance_heatmap.html")
print("    - 06_admission_volume_trend.html")
print("    - 07_treatment_effectiveness.html")
print("    - 09_kpi_dashboard.html")
print("\n  PNG (Publication Quality):")
print("    - 02_recovery_by_department_sns.png")
print("    - 03_los_trend_sns.png")
print("    - 04_los_by_department_box.png")
print("    - 05_department_performance_heatmap.png")
print("    - 06_admission_volume_trend_sns.png")
print("    - 07_treatment_effectiveness_sns.png")
print("    - 08_resource_utilization.png")
print("\n✓ Visualizations completed!")